In [2]:
!pip install -q zarr numcodecs xarray dask scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 140.9 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import xarray as xr
xr.backends.list_engines()


{'h5netcdf': <H5netcdfBackendEntrypoint>
   Open netCDF (.nc, .nc4 and .cdf) and most HDF5 files using h5netcdf in Xarray
   Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.H5netcdfBackendEntrypoint.html,
 'scipy': <ScipyBackendEntrypoint>
   Open netCDF files (.nc, .cdf and .nc.gz) using scipy in Xarray
   Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ScipyBackendEntrypoint.html,
 'store': <StoreBackendEntrypoint>
   Open AbstractDataStore instances in Xarray
   Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.StoreBackendEntrypoint.html,
 'zarr': <ZarrBackendEntrypoint>
   Open zarr files (.zarr) using zarr in Xarray
   Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ZarrBackendEntrypoint.html}

In [3]:
# ================================
# EXPERIMENTS & RESULTS NOTEBOOK
# ================================

from google.colab import drive
drive.mount('/content/drive')

import xarray as xr
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pandas as pd

# --------------------
# SETUP
# --------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

DATA_PATH = "/content/drive/MyDrive/Indian Weather Prediction/datasets/processed.zarr"
MODEL_PATH = "/content/drive/MyDrive/Indian Weather Prediction/models/convlstm_final.pt"

VARS = ["t2m", "d2m", "sp", "msl", "u10", "v10", "tp"]
INPUT_STEPS = 4

# --------------------
# MODEL (same as training)
# --------------------
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(7, 32, 3, padding=1)
        self.relu = nn.ReLU()
        self.temp_head = nn.Conv2d(32, 1, 1)
        self.rain_head = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        x = x[:, -1]
        x = self.relu(self.conv(x))
        return self.temp_head(x), self.rain_head(x)

model = ConvNet().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

# --------------------
# LOAD DATA
# --------------------
ds = xr.open_zarr(DATA_PATH)

TEST_YEARS = ["2015", "2016", "2017", "2018", "2022", "2023"]

results = []

# --------------------
# EXPERIMENT LOOP (MEMORY SAFE)
# --------------------
for year in TEST_YEARS:
    print(f"Running experiments for year {year}...")

    ds_y = ds.sel(valid_time=slice(f"{year}-01-01", f"{year}-12-31"))

    preds_t, true_t, base_t = [], [], []
    preds_r, true_r, base_r = [], [], []

    for i in range(INPUT_STEPS, len(ds_y.valid_time)):
        x = np.stack(
            [ds_y[v].isel(valid_time=slice(i-INPUT_STEPS, i)).values for v in VARS],
            axis=1
        )
        x = torch.tensor(x[None], dtype=torch.float32).to(device)

        with torch.no_grad():
            pt, pr = model(x)

        preds_t.append(pt.squeeze().cpu().numpy())
        preds_r.append(pr.squeeze().cpu().numpy())

        true_t.append(ds_y["t2m"].isel(valid_time=i).values)
        true_r.append(ds_y["tp"].isel(valid_time=i).values)

        base_t.append(ds_y["t2m"].isel(valid_time=i-1).values)
        base_r.append(ds_y["tp"].isel(valid_time=i-1).values)

    # Flatten
    preds_t = np.array(preds_t).flatten()
    preds_r = np.array(preds_r).flatten()
    true_t  = np.array(true_t).flatten()
    true_r  = np.array(true_r).flatten()
    base_t  = np.array(base_t).flatten()
    base_r  = np.array(base_r).flatten()

    # --------------------
    # METRICS
    # --------------------
    rmse_t = np.sqrt(mean_squared_error(true_t, preds_t))
    mae_t  = mean_absolute_error(true_t, preds_t)
    r2_t   = r2_score(true_t, preds_t)

    rmse_r = np.sqrt(mean_squared_error(true_r, preds_r))
    mae_r  = mean_absolute_error(true_r, preds_r)
    r2_r   = r2_score(true_r, preds_r)

    rmse_t_base = np.sqrt(mean_squared_error(true_t, base_t))
    rmse_r_base = np.sqrt(mean_squared_error(true_r, base_r))

    skill_t = 1 - (rmse_t / rmse_t_base)
    skill_r = 1 - (rmse_r / rmse_r_base)

    results.append([
        year,
        rmse_t, mae_t, r2_t, skill_t,
        rmse_r, mae_r, r2_r, skill_r
    ])

# --------------------
# RESULTS TABLE
# --------------------
columns = [
    "Year",
    "Temp_RMSE", "Temp_MAE", "Temp_R2", "Temp_Skill",
    "Rain_RMSE", "Rain_MAE", "Rain_R2", "Rain_Skill"
]

df = pd.DataFrame(results, columns=columns)

print("\nEXPERIMENT RESULTS (PER YEAR):\n")
print(df)

print("\nAVERAGE PERFORMANCE ACROSS YEARS:\n")
print(df.mean(numeric_only=True))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Running experiments for year 2015...
Running experiments for year 2016...
Running experiments for year 2017...
Running experiments for year 2018...
Running experiments for year 2022...
Running experiments for year 2023...

EXPERIMENT RESULTS (PER YEAR):

   Year  Temp_RMSE  Temp_MAE   Temp_R2  Temp_Skill  Rain_RMSE  Rain_MAE  \
0  2015   0.272309  0.193349  0.926270    0.123976   0.001087  0.000688   
1  2016   0.273354  0.193908  0.921815    0.132507   0.001092  0.000690   
2  2017   0.273064  0.195283  0.923515    0.128156   0.001118  0.000700   
3  2018   0.280676  0.200106  0.921711    0.118239   0.001119  0.000706   
4  2022   0.281809  0.201396  0.920314    0.112653   0.001112  0.000698   
5  2023   0.274479  0.195477  0.925023    0.131375   0.001072  0.000672   

    Rain_R2  Rain_Skill  
0 -2.851871   -0.805761  
1 -3.074581   -0.84